# BOI Cognidroid — DREBIN-215 Raw Feature Training v3

## What this does differently

| Old notebook | This notebook |
|---|---|
| Collapses 214 columns into 12 hardcoded counts | Feeds all 214 raw binary columns to XGBoost |
| YOU decide what is dangerous | MODEL learns from data |
| SYSTEM_ALERT_WINDOW = same risk as SEND_SMS | Model assigns each permission its own weight |

## Output files
1.  — trained model
2.  — exact column order for production inference

Both go into 


In [ ]:
!pip install -q kagglehub xgboost==2.0.3 scikit-learn shap pandas numpy matplotlib

In [ ]:
import kagglehub, os, glob
import pandas as pd
import numpy as np

dataset_path = kagglehub.dataset_download(
    "shashwatwork/android-malware-dataset-for-machine-learning"
)
print("Downloaded to:", dataset_path)
for f in os.listdir(dataset_path): print(" -", f)

In [ ]:
# Load the main feature matrix (large file) and categories lookup
all_csvs = glob.glob(f"{dataset_path}/**/*.csv", recursive=True)
csv_path = max([p for p in all_csvs if "categor" not in p.lower()],
               key=lambda p: sum(1 for _ in open(p)))
categories_path = next((p for p in all_csvs if "categor" in p.lower()), None)

df = pd.read_csv(csv_path)
cat_df = pd.read_csv(categories_path) if categories_path else None

label_col = df.columns[-1]
print(f"Shape: {df.shape}")
print(f"Label column: {label_col!r}")
print(df[label_col].value_counts())

In [ ]:
# IMPORTANT: Print ALL 214 column names so you can see what DREBIN features look like.
# This output tells you what format API call names are in (slash-separated vs dot-separated).
# The production feature_extractor.py MUST match these exact strings.
feature_cols = [c for c in df.columns if c != label_col]
print(f"Total feature columns: {len(feature_cols)}")
print(f"All binary 0/1: {df[feature_cols].isin([0,1]).all().all()}")
print("
=== ALL 214 COLUMN NAMES ===")
for i, col in enumerate(feature_cols):
    print(f"  [{i:3d}] {col}")

if cat_df is not None:
    name_col = cat_df.columns[0]
    category_col = cat_df.columns[1]
    print("
=== CATEGORY BREAKDOWN ===")
    print(cat_df[category_col].value_counts())
    print("
Sample Manifest Permissions:")
    for p in cat_df[cat_df[category_col]=="Manifest Permission"][name_col].tolist()[:20]:
        print(" ", p)
    print("
Sample API Calls:")
    for a in cat_df[cat_df[category_col]=="API call signature"][name_col].tolist()[:20]:
        print(" ", a)

In [ ]:
# Build category map: {column_name -> "Manifest Permission" / "API call signature" / etc}
import re
feature_categories = {}
if cat_df is not None:
    cat_map = dict(zip(cat_df[cat_df.columns[0]], cat_df[cat_df.columns[1]]))
    for col in feature_cols:
        feature_categories[col] = cat_map.get(col, "Unknown")
else:
    ip_re = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")
    for col in feature_cols:
        if ".permission." in col: feature_categories[col] = "Manifest Permission"
        elif "intent" in col.lower(): feature_categories[col] = "Intent"
        elif "/" in col: feature_categories[col] = "API call signature"
        elif ip_re.match(col): feature_categories[col] = "Network address"
        else: feature_categories[col] = "Other"

from collections import Counter
for cat, cnt in Counter(feature_categories.values()).most_common():
    print(f"  {cat:30s}: {cnt}")

In [ ]:
# Prepare X (214 raw binary features) and y (0=Benign, 1=Malicious)
X = df[feature_cols].astype(np.float32).values
print(f"X shape: {X.shape}")

label_vals = df[label_col].astype(str).str.strip().str.upper()
if label_vals.isin(["S","B"]).all():
    y = (label_vals == "S").astype(int).values
    print("Labels: S=Malicious, B=Benign")
elif label_vals.isin(["1","0"]).all():
    y = label_vals.astype(int).values
    print("Labels: 1=Malicious, 0=Benign")
else:
    y = (~label_vals.isin(["B","0","BENIGN"])).astype(int).values
    print("Labels decoded via fallback")

print(f"Malicious: {y.sum():,}  Benign: {(y==0).sum():,}")

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
w_train = compute_sample_weight("balanced", y_train)

dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, feature_names=feature_cols)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=feature_cols)

params = {
    "objective": "binary:logistic",
    "max_depth": 6, "eta": 0.1,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "eval_metric": "auc", "seed": 42,
}

print(f"Training on {X_train.shape[1]} raw DREBIN features (no hardcoding)...")
model = xgb.train(
    params, dtrain, num_boost_round=500,
    evals=[(dtrain,"train"),(dtest,"test")],
    early_stopping_rounds=40, verbose_eval=50,
)
print(f"Best iteration: {model.best_iteration}")

In [ ]:
proba = model.predict(dtest, iteration_range=(0, model.best_iteration+1))
y_pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["Benign","Malicious"]))
print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")

In [ ]:
# Score distribution sanity check
# Production formula: risk_score = p_malicious * 90
import matplotlib.pyplot as plt

scores = proba * 90.0
benign_scores = scores[y_test == 0]
mal_scores    = scores[y_test == 1]

print("=== Score Distribution (p_mal * 90) ===")
print(f"Benign    mean:{benign_scores.mean():5.1f}  p90:{np.percentile(benign_scores,90):5.1f}  WANT mean<10")
print(f"Malicious mean:{mal_scores.mean():5.1f}  p10:{np.percentile(mal_scores,10):5.1f}  WANT mean>75")

plt.figure(figsize=(10,4))
plt.hist(benign_scores, bins=50, alpha=0.6, label="Benign", color="green")
plt.hist(mal_scores,    bins=50, alpha=0.6, label="Malicious", color="red")
plt.axvline(35, color="orange", linestyle="--", label="Safety cap (35)")
plt.axvline(75, color="darkred", linestyle="--", label="Highly Malicious (75)")
plt.xlabel("Risk Score"); plt.ylabel("Count"); plt.title("Score Distribution")
plt.legend(); plt.tight_layout()
plt.savefig("score_distribution.png", dpi=150)
plt.show()

In [ ]:
# Top 30 features by importance — what the MODEL considers dangerous
importance = model.get_score(importance_type="gain")
imp_df = (pd.DataFrame(list(importance.items()), columns=["feature","gain"])
          .sort_values("gain", ascending=False).head(30))
imp_df["category"] = imp_df["feature"].map(feature_categories)
print("=== Top 30 Features (model-learned importance) ===")
print(imp_df.to_string(index=False))

zero = [f for f in feature_cols if f not in importance]
print(f"
Zero-importance (model ignores these entirely): {len(zero)}")
for f in zero[:15]: print(" ", f)

In [ ]:
# Save model + feature names — BOTH files needed for deployment
import json

model.save_model("xgb_risk_model.json")
print("Saved: xgb_risk_model.json")

meta = {
    "feature_names":     feature_cols,
    "feature_categories": feature_categories,
    "n_features":        len(feature_cols),
    "model_type":        "binary:logistic",
    "class_names":       ["Benign", "Malicious"],
    "score_formula":     "p_malicious * 90.0",
    "best_iteration":    int(model.best_iteration),
    "train_auc":         float(roc_auc_score(y_test, proba)),
}
with open("drebin_feature_names.json","w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved: drebin_feature_names.json ({len(feature_cols)} features)")

try:
    from google.colab import files
    files.download("xgb_risk_model.json")
    files.download("drebin_feature_names.json")
    files.download("score_distribution.png")
    print("Downloads triggered!")
except ImportError:
    print("Not in Colab — files saved locally")

## Deploy



**Expected results after deploy:**
- Calculator: score < 10 (Safe) — has no SEND_SMS, no BIND_ACCESSIBILITY_SERVICE
- Banking trojan: score > 75 (Highly Malicious)
- No more hardcoded dangerous list anywhere in the codebase
